# 06 · Monitoramento

*O modelo continua bom? As recomendações continuam diversas?*

> **FIAP Pós-Tech · Fase 5 (MLOps)** — Recomendação de notícias com MLOps no Databricks.
> Matérias **reais** (sitemap do O Globo) · usuários e cliques **sintéticos**.

## O que monitorar num recomendador

| Sinal | Pergunta | Como medimos |
|---|---|---|
| **Qualidade offline** | o modelo novo é tão bom quanto os anteriores? | tendência do NDCG@10 (`model_metrics`) |
| **Cobertura** | estamos recomendando sempre as mesmas 20 matérias? | % do catálogo recomendado |
| **Drift / bolha** | recomendamos o que as pessoas leem, ou só política? | **PSI** entre editorias lidas × recomendadas |
| **Qualidade online** | as pessoas clicam? | CTR em produção / teste A/B (fora do escopo) |

PSI (*Population Stability Index*): < 0,1 estável · 0,1–0,2 atenção · > 0,2 mudança relevante.

In [ ]:
# Célula 1 — dependências (sempre a primeira: o restartPython apaga as variáveis)
%pip install -q -r ../requirements-databricks.txt
dbutils.library.restartPython()

In [ ]:
# Célula 2 — src/ no sys.path + widgets (= parâmetros do Job) -> settings
import sys
from pathlib import Path


def _find_src() -> Path:
    candidates = [Path.cwd().parent / "src"]
    try:  # fallback: caminho do notebook no workspace
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        candidates.append(Path("/Workspace" + ctx.notebookPath().get()).parent.parent / "src")
    except Exception:  # noqa: BLE001
        pass
    for c in candidates:
        if (c / "news_recsys").is_dir():
            return c.resolve()
    raise FileNotFoundError(f"src/ não encontrado em {candidates}")


SRC = _find_src()
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
ROOT = SRC.parent

from news_recsys.config import Settings

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "news_recsys_dev")
dbutils.widgets.text("n_users", "300")
dbutils.widgets.text("seed", "42")

settings = Settings(_env_file=None).with_overrides(
    catalog=dbutils.widgets.get("catalog"),
    schema_name=dbutils.widgets.get("schema"),
    n_users=dbutils.widgets.get("n_users"),
    seed=dbutils.widgets.get("seed"),
)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {settings.catalog}.{settings.schema_name}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {settings.catalog}.{settings.schema_name}.{settings.volume}")
print(f"schema ..... {settings.catalog}.{settings.schema_name}")
print(f"modelo ..... {settings.model_fqn}")
print(f"usuários ... {settings.n_users} (seed={settings.seed})")

In [ ]:
# Dados de entrada: catálogo (bronze) + usuários e cliques (silver)
import pandas as pd

articles = spark.table(settings.table("bronze_articles")).toPandas()
window = spark.table(settings.table("silver_catalog_window")).toPandas()["article_id"]
articles = articles[articles["article_id"].isin(window)].reset_index(drop=True)
users = spark.table(settings.table("silver_users")).toPandas()
clicks = spark.table(settings.table("silver_interactions")).toPandas()
clicks["event_ts"] = pd.to_datetime(clicks["event_ts"], utc=True)
print(f"{len(articles)} matérias · {len(users)} usuários · {len(clicks)} cliques")

### `monitoring.py`

In [ ]:
# src/news_recsys/monitoring.py  (somente leitura — o código de verdade está em src/)
"""Monitoramento do recomendador: distribuição de editorias, PSI (drift) e cobertura."""

from __future__ import annotations

from collections.abc import Sequence
from datetime import UTC, datetime
from typing import Any

import numpy as np
import pandas as pd

PSI_ALERT = 0.2  # regra de bolso: < 0.1 estável · 0.1–0.2 atenção · > 0.2 mudança relevante


def distribution(values: Sequence[str] | pd.Series) -> pd.Series:
    s = pd.Series(list(values), dtype="object")
    return s.value_counts(normalize=True).sort_index() if len(s) else pd.Series(dtype=float)


def psi(expected: pd.Series, actual: pd.Series, eps: float = 1e-4) -> float:
    """Population Stability Index entre duas distribuições categóricas."""
    cats = sorted(set(expected.index) | set(actual.index))
    e = expected.reindex(cats).fillna(0).to_numpy() + eps
    a = actual.reindex(cats).fillna(0).to_numpy() + eps
    e, a = e / e.sum(), a / a.sum()
    return float(np.sum((a - e) * np.log(a / e)))


def recs_report(
    recs: pd.DataFrame, interactions: pd.DataFrame, articles: pd.DataFrame
) -> dict[str, float]:
    """Compara o que recomendamos com o que os usuários consomem."""
    section = articles.set_index("article_id")["section"]
    rec_dist = distribution(recs["article_id"].map(section).dropna())
    read_dist = distribution(interactions["article_id"].map(section).dropna())
    n_catalog = len(articles)
    return {
        "n_users": float(recs["user_id"].nunique()),
        "n_recommendations": float(len(recs)),
        "catalog_coverage": recs["article_id"].nunique() / n_catalog if n_catalog else 0.0,
        "psi_sections_recs_vs_reads": psi(read_dist, rec_dist),
        "top_section_share": float(rec_dist.max()) if len(rec_dist) else 0.0,
    }


def report_to_records(
    report: dict[str, float], *, model_version: str, generated_at: str | None = None
) -> list[dict[str, Any]]:
    ts = generated_at or datetime.now(UTC).isoformat(timespec="seconds")
    return [
        {"generated_at": ts, "model_version": model_version, "metric": k, "value": float(v)}
        for k, v in report.items()
    ]

### Execução: relatório desta execução → `recs_monitoring`

In [ ]:
from news_recsys.monitoring import PSI_ALERT, distribution, recs_report, report_to_records

gold = spark.table(settings.table("gold_user_recommendations")).toPandas()
report = recs_report(gold, clicks, articles)
version = str(gold["model_version"].iloc[0])
spark.createDataFrame(pd.DataFrame(report_to_records(report, model_version=version))).write.mode("append").saveAsTable(
    settings.table("recs_monitoring")
)
status = "ALERTA" if report["psi_sections_recs_vs_reads"] > PSI_ALERT else "ok"
print(f"PSI editorias (lidas × recomendadas) = {report['psi_sections_recs_vs_reads']:.3f} -> {status}")
pd.Series(report).round(4).to_frame("valor")

### Visualizando: o que as pessoas leem × o que recomendamos

In [ ]:
import matplotlib.pyplot as plt

section = articles.set_index("article_id")["section"]
dist = pd.DataFrame(
    {"lidas": distribution(clicks["article_id"].map(section).dropna()), "recomendadas": distribution(gold["section"].dropna())}
).fillna(0).sort_values("lidas", ascending=True)
ax = dist.plot.barh(figsize=(8, 4.5))
ax.set_title("Distribuição por editoria")
ax.set_xlabel("fração")
plt.tight_layout()
display(ax.figure)

### Execução: tendência do campeão ao longo dos dias

In [ ]:
mm = spark.table(settings.table("model_metrics")).toPandas()
trend = mm[(mm["metric"] == "ndcg_at_k") & (mm["promoted"])].sort_values("evaluated_at")
if len(trend) > 1:
    ax = trend.plot(x="evaluated_at", y="value", marker="o", figsize=(8, 3.5), legend=False, title="NDCG@10 do campeão")
    plt.tight_layout()
    display(ax.figure)
mm.pivot_table(index=["evaluated_at", "model_version"], columns="metric", values="value", aggfunc="max").tail(5).round(4)

### Um alerta em SQL (Databricks SQL → Alerts)

```sql
SELECT value AS psi
FROM workspace.news_recsys.recs_monitoring
WHERE metric = 'psi_sections_recs_vs_reads'
ORDER BY generated_at DESC
LIMIT 1
-- condição: psi > 0.2  → e-mail para o time
```

## Fechando o ciclo de MLOps

```
dados novos (GitHub Actions) → bronze → silver → treino (MLflow) → gate → @champion
      ↑                                                                     │
      └──── monitoramento ◄──── gold ◄──── inferência batch ◄──────────────┘
```

### O que explorar a seguir

- **Teste A/B** online: CTR do campeão × desafiante com tráfego real.
- **Modelos híbridos** (conteúdo + colaborativo) para o *cold start* de matérias.
- **Feature Store** e **Model Serving** para recomendação em tempo real.
- **Lakehouse Monitoring** e dashboards AI/BI sobre `recs_monitoring`.